<img src="images/panel_logo.png" width="10%" align="right">

# Big Data Application

In this notebook we are going to put our visualizations together into an application using Panel Pipelines.

---

In [1]:
!pip install dask-gateway
!pip install param
!pip install panel
!pip install hvplot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 14.7 MB/s eta 0:00:00


In [ ]:
!pip install jupyter_bokeh


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.8 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [2]:
import param
import panel as pn

pn.extension()

/tmp/ipykernel_1103/1488784431.py:4: UserWarning: Using Panel interactively in Colab notebooks requires the jupyter_bokeh package to be installed. Install it with:

    !pip install jupyter_bokeh

and try again.
  pn.extension()


In [3]:
import dask_gateway
import dask.dataframe as dd

import hvplot.dask

In [7]:
import os
import io
import zipfile
import requests

csv_path = (
    "/content/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "(1987_present)_2022_1.csv"
)

if not os.path.exists(csv_path):

    url = (
        "https://transtats.bts.gov/PREZIP/"
        "On_Time_Reporting_Carrier_On_Time_Performance_"
        "1987_present_2022_1.zip"
    )

    r = requests.get(url)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_file = [
            f for f in z.namelist()
            if f.endswith(".csv")
        ][0]

        z.extract(csv_file, "/content")

        csv_path = os.path.join(
            "/content",
            csv_file
        )

print("File dữ liệu:")
print(csv_path)

File dữ liệu:
/content/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv


In [4]:
from dask.distributed import Client, LocalCluster

# Khởi tạo cluster tính toán ngay trên RAM/CPU của Google Colab
cluster = LocalCluster()
client = Client(cluster)

# Xem thông tin Client / Dashboard
client

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:33339
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46209'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39087'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:34613 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:34613
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:38278
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:46043 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:46043
INFO:distributed.core:Starting established connection to tcp://127

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33339,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46043,Total threads: 1
Dashboard: http://127.0.0.1:36923/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:46209,


In [5]:
from dask.distributed import Client, LocalCluster

class LaunchDaskCluster(param.Parameterized):
    workers = pn.widgets.IntRangeSlider(name='Number of Workers', start=1, end=10, value=(1, 5), step=1)
    launch_dask = pn.widgets.Button(name='Launch Dask', button_type='primary')
    dask_status = pn.widgets.StaticText(name='Dask Status', value='Not Connected')
    years = pn.widgets.IntRangeSlider(name='Choose Date Range', start=2003, end=2022, value=(2018, 2022), step=1)

    cluster = None
    cluster_name = param.String()
    client = None
    ready = param.Boolean(default=False, precedence=-1)

    def __init__(self, **params):
        super().__init__(**params)
        self.launch_dask.on_click(self.dask_launcher)

    @param.output(('cluster_name', param.String), ('start_year', param.Number), ('end_year', param.Number))
    def output(self):
        return self.cluster_name, self.years.value[0], self.years.value[1]

    def dask_launcher(self, event):
        self.dask_status.value = "Launching Local Dask Cluster..."

        # Khởi tạo Local Cluster trực tiếp trên Colab
        if self.cluster is None:
            self.cluster = LocalCluster(n_workers=self.workers.value[1])
            self.client = Client(self.cluster)

        self.cluster_name = "LocalCluster"
        self.dask_status.value = f"Cluster Ready - {self.client.dashboard_link}"
        self.ready = True

    def panel(self):
        return pn.Column(
            self.workers,
            self.years,
            self.launch_dask,
            self.dask_status,
        )

stage1 = LaunchDaskCluster()
stage1.panel()

Column
    [0] IntRangeSlider(end=10, label='Number of Workers', name='Number of Workers', start=1, value=(1, 5), value_end=5, value_start=1)
    [1] IntRangeSlider(end=2022, label='Choose Date Range', name='Choose Date Range', start=2003, value=(2018, 2022), value_end=2022, value_start=2018)
    [2] Button(button_type='primary', color='primary', label='Launch Dask', name='Launch Dask')
    [3] StaticText(label='Dask Status', name='Dask Status', value='Not Connected')

In [8]:
import param
import panel as pn
import dask.dataframe as dd
import hvplot.dask
from google.colab import output

class Dashboard(param.Parameterized):
    # 1. Bộ Widget điều khiển
    method = pn.widgets.RadioButtonGroup(name='Method', options=['min', 'mean', 'max'], value='mean')
    field = pn.widgets.RadioButtonGroup(name='Field', options=['DepDelay', 'ArrDelay'], value='ArrDelay')
    groupby = pn.widgets.RadioButtonGroup(
        name='GroupBy',
        options=['Year', 'Month', 'DayofMonth', 'Reporting_Airline'],
        value='Month'
    )

    cluster_name = param.String(default="LocalCluster")
    start_year = param.Number(default=2022)
    end_year = param.Number(default=2022)

    def __init__(self, **params):
        super().__init__(**params)

        # 2. Bộ cột chuẩn theo dataset On_Time_Reporting_Carrier_2022
        cols = [
            'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline',
            'Origin', 'Dest', 'CRSDepTime', 'DepTime', 'DepDelay', 'ArrTime', 'ArrDelay',
            'Cancelled', 'Diverted', 'Distance'
        ]

        # Đọc dữ liệu CSV bằng Dask
        df = dd.read_csv(
            'On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv',
            usecols=cols,
            assume_missing=True
        )

        # Lọc dữ liệu theo năm và chuyến bay thực hiện (không hủy, không đổi hướng)
        df_filtered = df[(df['Year'] >= self.start_year) & (df['Year'] <= self.end_year)]
        self.flights = df_filtered[(df_filtered['Cancelled'] == 0) & (df_filtered['Diverted'] == 0)]

    @param.depends('cluster_name')
    def dask_dashboard(self):
        from dask.distributed import Client
        try:
            client = Client.current()
            # Trích xuất port từ dashboard link (ví dụ: 36763)
            port = client.dashboard_link.split(':')[-1].split('/')[0]
            # Tạo đường dẫn Proxy tương thích với Google Colab
            url = output.eval_js(f'google.colab.kernel.proxyPort({port})')
        except Exception:
            url = "#"

        return pn.pane.HTML(f"""
            <div style="padding: 20px; border: 1px solid #e0e0e0; border-radius: 8px; background-color: #f9f9f9; text-align: center; margin-top: 20px;">
                <h3 style="color: #2c3e50; margin-bottom: 10px;">Dask Cluster Monitor</h3>
                <p style="color: #27ae60; font-size: 14px;">Trạng thái: <b>{self.cluster_name}</b></p>
                <a href="{url}" target="_blank" style="display: inline-block; padding: 10px 20px; background-color: #2980b9; color: white; text-decoration: none; border-radius: 5px; font-weight: bold;">
                    🚀 Mở Dask Dashboard (Colab Proxy)
                </a>
            </div>
        """, width=350)

    @param.depends('groupby.value', 'method.value', 'field.value')
    def plot_data(self):
        # Gom nhóm và tính toán tổng hợp
        data = (
            self.flights
                .groupby(self.groupby.value)[self.field.value]
                .agg(self.method.value)
                .reset_index()
        )

        # Vẽ biểu đồ cột bằng hvPlot
        return data.hvplot.bar(
            x=self.groupby.value,
            y=self.field.value,
            title=f"Thống kê {self.field.value} ({self.method.value}) theo {self.groupby.value}",
            ylabel=self.field.value,
            height=400,
            width=600
        )

    # --- BỔ SUNG PHƯƠNG THỨC OUTPUT ĐỂ TRUYỀN DỮ LIỆU SANG STAGE 3 ---
    def output(self):
        return self.cluster_name

    def panel(self):
        return pn.Row(
            pn.Column(
                self.groupby,
                self.field,
                self.method,
                pn.layout.Divider(),
                self.plot_data,
            ),
            self.dask_dashboard,
        )

# Khởi tạo Stage 2 nhận đầy đủ 3 giá trị từ stage1.output()
cluster_nm, start_yr, end_yr = stage1.output()
stage2 = Dashboard(
    cluster_name=cluster_nm,
    start_year=start_yr,
    end_year=end_yr
)
stage2.panel()

Row
    [0] Column
        [0] RadioButtonGroup(label='GroupBy', name='GroupBy', options=['Year', 'Month', ...], value='Month')
        [1] RadioButtonGroup(label='Field', name='Field', options=['DepDelay', 'ArrDelay'], value='ArrDelay')
        [2] RadioButtonGroup(label='Method', name='Method', options=['min', 'mean', 'max'], value='mean')
        [3] Divider()
        [4] ParamMethod(method, _pane=HoloViews, defer_load=False)
    [1] ParamMethod(method, _pane=HTML, defer_load=False)

In [10]:
stage2 = Dashboard(
            cluster_name=stage1.output()[0],
            start_year=stage1.output()[1],
            end_year=stage1.output()[2]
)
# stage2.panel()

In [9]:
import param
import panel as pn
from dask.distributed import Client

class StopDaskCluster(param.Parameterized):
    cluster_name = param.String()
    dask_status = pn.widgets.StaticText(name='Dask Status', value='Connected')
    stop_dask = pn.widgets.Button(name='Stop Dask Cluster', button_type='danger')

    def __init__(self, **params):
        super().__init__(**params)
        self.stop_dask.on_click(self.shutdown_cluster)

    @param.depends('cluster_name', watch=True)
    def check_status(self):
        try:
            client = Client.current()
            self.dask_status.value = f"Cluster Status: {client.status.title()}"
        except Exception:
            self.dask_status.value = "Cluster Status: Closed / Not Found"

    def shutdown_cluster(self, event):
        self.dask_status.value = "Shutting Cluster Down..."
        try:
            client = Client.current()
            cluster = client.cluster

            # Đóng client và cluster theo đúng thứ tự
            client.close()
            if cluster is not None:
                cluster.close()

            self.dask_status.value = "Cluster Status: Closed Successfully"
        except Exception:
            self.dask_status.value = "Cluster Status: Already Closed or Not Found"

    def panel(self):
        return pn.Column(
            self.stop_dask,
            self.dask_status,
        )

# Khởi tạo và hiển thị Stage 3
stage3 = StopDaskCluster(cluster_name=stage2.output())
stage3.panel()

Column
    [0] Button(button_type='danger', color='danger', label='Stop Dask Cluster', name='Stop Dask Cluster')
    [1] StaticText(label='Dask Status', name='Dask Status', value='Connected')

In [11]:
stage3 = StopDaskCluster(cluster_name=stage2.output())
# stage3.panel()

In [12]:
pipeline = pn.pipeline.Pipeline()

In [13]:
pipeline.add_stage('Launch Dask', stage1)
pipeline.add_stage('Visualization', stage2)
pipeline.add_stage('Stop Cluster', stage3)

In [14]:
pipeline.servable()

Column(sizing_mode='stretch_width')
    [0] Row(sizing_mode='stretch_width')
        [0] Column
            [0] Markdown(str, margin=(0, 0, 0, 5))
            [1] Row(width=100)
        [1] HoloViews(Overlay, backend='bokeh', height=80, sizing_mode='stretch_width')
        [2] Row
            [0] Button(disabled=True, label='Previous', name='Previous', width=125)
            [1] Button(label='Next', name='Next', width=125)
    [1] Row
        [0] Column
            [0] IntRangeSlider(end=10, label='Number of Workers', name='Number of Workers', start=1, value=(1, 5), value_end=5, value_start=1)
            [1] IntRangeSlider(end=2022, label='Choose Date Range', name='Choose Date Range', start=2003, value=(2018, 2022), value_end=2022, value_start=2018)
            [2] Button(button_type='primary', color='primary', label='Launch Dask', name='Launch Dask')
            [3] StaticText(label='Dask Status', name='Dask Status', value='Not Connected')

In [15]:
gateway.list_clusters()

NameError: name 'gateway' is not defined

---

## Next →

We'll discuss some ways to [collaborate on data science projects](./09-collaborative-data-science.ipynb).